In [ ]:
import sys
import os
sys.path.append(os.path.abspath('Multi-Agent-Initialization'))

# Structured Web Data Extraction Agent - Version 4

This version builds upon V3 by introducing a **Critic / Quality Control Node**. 
Before the final answer is shown to the user, a Critic agent reviews the synthesized response against the raw extracted context to ensure there are no hallucinations, no missing critical facts, and adherence to the structured guidelines. If it fails, the Synthesizer attempts a rewrite.

In [ ]:
import os
import json
import warnings
import re
import requests
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

from typing import Annotated, Literal, Optional, List, Dict, Any
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, START, StateGraph, END
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

import prompts

# Setup LLMs
reasoning_llm = ChatGroq(model='llama-3.1-8b-instant')
llm = reasoning_llm

# Specialized State for Data Extracting Pipeline
class State(MessagesState):
    enabled_agents: Optional[List[str]]
    plan: Optional[Dict[str, Dict[str, Any]]]
    user_query: Optional[str]
    current_step: int
    replan_flag: Optional[bool]
    last_reason: Optional[str]
    replan_attempts: Optional[Dict[int, int]]
    agent_query: Optional[str]
    final_answer: Optional[str]
    context_relevance_score: Optional[float]
    critic_feedback: Optional[str]
    critic_approved: Optional[bool]

In [ ]:
tavily_tool = TavilySearchResults(max_results=5, search_depth='advanced', include_raw_content=True)

web_search_agent = create_react_agent(
    llm,
    tools=[tavily_tool],
    prompt=prompts.agent_system_prompt('You are the Web Data Extraction Researcher.')
)

def planner_node(state: State) -> Command[Literal['executor']]:
    plan_instructions = """You are the Planner. Determine a clear plan to fulfill the user query.
Generate a JSON output mapping each step "1", "2", etc., to an object with:
"agent": (web_researcher or synthesizer or critic)
"goal": "string"

User Query: """ + state.get("user_query", "")

    llm_reply = reasoning_llm.invoke([HumanMessage(content=plan_instructions)])
    
    try:
        content_str = llm_reply.content if isinstance(llm_reply.content, str) else str(llm_reply.content)
        content_str = content_str.replace('```json', '').replace('```', '')
        parsed_plan = json.loads(content_str)
    except Exception as e:
        parsed_plan = {
            "1": {"agent": "web_researcher"},
            "2": {"agent": "synthesizer"},
            "3": {"agent": "critic"}
        }

    return Command(
        update={
            "plan": parsed_plan,
            "messages": [HumanMessage(content=json.dumps(parsed_plan), name="initial_plan")],
            "user_query": state.get("user_query", state.get("messages", [{}])[0].content if state.get("messages") else ""),
            "current_step": 1,
            "replan_flag": False,
            "last_reason": "",
        },
        goto="executor",
    )

def executor_node(state: State) -> Command[Literal["web_researcher", "synthesizer", "critic", "planner"]]:
    plan = state.get("plan", {})
    step = state.get("current_step", 1)
    
    replan_flag = state.get("replan_flag", False)
    if replan_flag:
        print("EXECUTOR DETECTED A REPLAN FLAG! Autonomously replanning...")
        return Command(
            update={
                "replan_flag": False,
                "messages": [HumanMessage(content="Retrying workflow.", name="executor")],
                "current_step": 1
            },
            goto="planner"
        )
    
    planned_agent = plan.get(str(step), {}).get("agent", "critic")
    if planned_agent not in ["web_researcher", "synthesizer", "critic", "planner"]:
        planned_agent = "critic" if step == len(plan) else "synthesizer"

    return Command(
        update={
            "messages": [HumanMessage(content=f"Routing to {planned_agent} for step {step}", name="executor")],
            "current_step": step + 1,
        },
        goto=planned_agent
    )

def evaluate_context_relevance(query: str, context: str) -> float:
    eval_prompt = f"""
    Evaluate the relevance of the extracted text to the main query.
    Return ONLY a float score between 0.0 (empty) and 1.0 (highly relevant).
    
    Query: {query}
    Context excerpt: {context[:2000]}
    """
    try:
        reply = reasoning_llm.invoke([HumanMessage(content=eval_prompt)])
        score_str = reply.content.strip()
        match = re.search(r'0\.\d+|1\.0', score_str)
        if match: return float(match.group())
        return 0.5
    except:
        return 0.5

def web_research_node(state: State) -> Command[Literal["executor"]]:
    query = state.get("user_query", "")
    print(f"WEB SEARCH -> Querying: {query[:50]}...")
    
    final_content = ""
    url_match = re.search(r"https?://[^\s]+", query)
    target_url = url_match.group(0) if url_match else None
    
    if target_url and "reddit.com" in target_url:
        try:
            json_url = target_url.rstrip("/") + ".json?limit=1000"
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(json_url, headers=headers)
            if response.status_code == 200:
                data = response.json()
                post_data = data[0]['data']['children'][0]['data']
                title = post_data.get('title', '')
                selftext = post_data.get('selftext', '')
                comments = []
                if len(data) > 1:
                    for child in data[1]['data'].get('children', []):
                        if child.get('kind') == 't1':
                            body = child['data'].get('body', '').strip()
                            if body and body not in ["[deleted]", "[removed]"]:
                                comments.append(f"Comment: {body}")
                
                combined = "\n---\n".join(comments)
                final_content = f"Title: {title}\n\nPost: {selftext}\n\nComments:\n{combined[:50000]}"
        except Exception as e:
            pass
    
    if not final_content:
        try:
            agent_result = web_search_agent.invoke({"messages": [HumanMessage(content=query)]})
            final_content = agent_result["messages"][-1].content
        except Exception as e:
            final_content = f"Failed: {e}"
        
    relevance = evaluate_context_relevance(query, final_content)
    trigger_replan = relevance < 0.4
    
    return Command(
        update={
            "messages": [HumanMessage(content=final_content, name="web_researcher")],
            "context_relevance_score": relevance,
            "replan_flag": trigger_replan
        },
        goto="executor"
    )

def synthesizer_node(state: State) -> Command[Literal["executor"]]:
    print("SYNTHESIZER -> Drafting report...")
    relevant_msgs = [m.content for m in state.get("messages", []) if getattr(m, "name", None) == "web_researcher"]
    feedback = state.get("critic_feedback", "")
    
    prompt_add = f"\n\nCRITIC FEEDBACK (Address this immediately):\n{feedback}" if feedback else ""
    prompt = f"User question: {state.get('user_query', '')}\n\nContext:\n\n" + "\n".join(relevant_msgs) + prompt_add
    
    llm_reply = reasoning_llm.invoke([HumanMessage(content=prompt[:60000])])
    answer = llm_reply.content if isinstance(llm_reply.content, str) else str(llm_reply.content)
    
    return Command(update={"final_answer": answer.strip(), "messages": [HumanMessage(content=answer.strip(), name="synthesizer")]}, goto="executor")

def critic_node(state: State) -> Command[Literal["synthesizer", END]]:
    print("CRITIC -> Validating the report directly...")
    final_answer = state.get("final_answer", "")
    query = state.get("user_query", "")
    
    eval_prompt = f"""
    You are a strict QA Reviewer. Review the synthesized output for:
    1. Accuracy (no hallucinations, actually answers the query)
    2. Structure (is it well formatted in sections?)
    User Query: {query}
    Synthesized Answer: {final_answer}
    
    If it perfectly passes, output EXACTLY the word 'APPROVED'.
    If it fails, provide detailed feedback on what must be rewritten.
    """
    
    reply = reasoning_llm.invoke([HumanMessage(content=eval_prompt)])
    eval_result = reply.content.strip()
    
    if 'APPROVED' in eval_result.upper():
        print("✅ CRITIC APPROVED REPORT!")
        return Command(update={"critic_approved": True}, goto=END)
    else:
        print("⚠️ CRITIC REJECTED! Sending back to Synthesizer...")
        return Command(update={"critic_approved": False, "critic_feedback": eval_result}, goto="synthesizer")

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

workflow = StateGraph(State)
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("web_researcher", web_research_node)
workflow.add_node("synthesizer", synthesizer_node)
workflow.add_node("critic", critic_node)

workflow.add_edge(START, "planner")

memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

In [ ]:
target_url = "https://www.reddit.com/r/ClaudeCode/comments/1r5gk1d/40_days_of_vibe_coding_taught_me_the_most/"

query = f"Please thoroughly extract the main points, insights, and key takeaways from the following Reddit post URL:\n{target_url}"

state = {
    "messages": [HumanMessage(content=query)],
    "user_query": query,
}

config = {"configurable": {"thread_id": "v4_critic_run"}}

print("🚀 Starting V4 Pipeline (With Critic QA Node)...")
result = graph.invoke(state, config=config)

print("\n--- 📝 V4 FINAL VERIFIED SYNTHESIZED REPORT ---\n")
print(result.get("final_answer"))